# Importing files


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**pandas** is a Python library for data manipulation and analysis. It provides data structures and operations for working with numerical tables and time series.

In this notebook, you will learn how to load a dataset and explore, slice, clean, and manipulate it with pandas.

## 1.1 Installing and Importing Pandas

Run the cell below to install pandas if it isn't already installed in your environment. If it's already installed, this cell has no effect.

In [ ]:
#!pip install pandas

Import pandas under the conventional alias `pd`.

In [ ]:
import pandas as pd

## 1.2 Importing a Dataset

This notebook uses `titanic.csv`, a dataset describing the passengers of the Titanic (age, sex, ticket class, whether they survived, etc.).

Run the cell below and upload `titanic.csv` (found in this notebook's folder in the repo) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select titanic.csv

Load the uploaded CSV into a pandas DataFrame and print it.

In [ ]:
df = pd.read_csv('data.csv')
print(df)

`display()` renders a DataFrame more nicely than `print()` in a notebook.

In [ ]:
display(df)

# 1. Laboratory Objectives

• Prepare a real-world breast cancer classification dataset for supervised machine learning.

• Perform a train-test split before fitting preprocessing and feature-selection procedures.

• Apply data preprocessing without introducing information leakage from the test set.

• Conduct exploratory data analysis to understand distributions, class composition, and feature relationships.

• Apply and compare multiple feature-selection techniques.

• Train and compare several supervised machine-learning classifiers.

• Perform hyperparameter optimization using cross-validation on the training data.

• Evaluate classification models using complementary performance metrics and visualizations.

• Identify the most effective combination of feature-selection technique and classifier.

# Step 1: Load the Dataset

inspect the raw input to verify its format, dimensionality, and data integrity

In [ ]:
import pandas as pd

# 1. Read the CSV file into a DataFrame
df = pd.read_csv('data.csv')

# 2. Display the first few records
print("First 5 records:")
display(df.head())

# 3. Check the dataset dimensions (rows, columns)
print(f"\nDataset Shape: {df.shape}")

# 4. Inspect the column headers
print("\nColumn Names:")
print(df.columns.tolist())

# 5. Review data types, non-null counts, and memory usage
print("\nDataset Info:")
df.info()

df.info():
- Confirms that all 30 diagnostic features are numeric (float64), id is an integer (int64), diagnosis is a string/categorical label (object), and Unnamed: 32 contains zero non-null values. This validates that no numbers were misread as text.

 # Step 2: Inspect Data Quality
 auditing the dataset for anomalies (missing attributes, duplicate measurements, non-informative metadata, and wide scale variations)

 The four primary tasks of Step 2 address specific experimental requirements:   
 - Check for Missing Values: Scikit-learn estimators cannot process NaN values directly. Inspecting missingness confirms whether imputation is strictly necessary or whether entire empty columns exist.  
 - Check for Duplicate Records: Identical feature vectors from repeated patient entries artificially inflate performance metrics and induce cross-partition leakage if duplicates fall into both training and testing splits.   
 - Generate Descriptive Statistics: Evaluating distributional metrics (mean, standard deviation, quartiles, and extremes) reveals <mark>feature scale discrepancies</mark> (e.g., area_mean reaching values $> 2500$ while smoothness_mean remains $< 0.17$) and flags potential outliers.   
 - Identify Non-Predictive Columns: Identifies arbitrary artifacts such as id (which can cause memorization rather than learning morphometric patterns) and trailing empty columns like Unnamed: 32.

Feature scale discrepancy:
  - Feature scale discrepancies refer to situations where different input variables in a dataset have vastly different numerical ranges, magnitudes, and units of measurement.   In the Breast Cancer Wisconsin dataset, this discrepancy is immediately apparent when comparing geometric features derived from tumor cell nuclei

In [ ]:
import pandas as pd

# Load dataset (from Step 1)
df = pd.read_csv('data.csv')

# Task A: Check for missing values across all columns
missing_counts = df.isnull().sum()
columns_with_missing = missing_counts[missing_counts > 0]
print("Columns with Missing Values:")
print(columns_with_missing if not columns_with_missing.empty else "No missing values found.")

# Task B: Check for duplicate records
num_duplicates = df.duplicated().sum()
# Computes row-wise hash equivalence across all feature vectors.
print(f"\nTotal Duplicate Records: {num_duplicates}")

# Task C: Generate descriptive summary statistics for numerical features
print("\nDescriptive Statistics (Selected Numerical Features):")
display(df.describe())

# Task D: Inspect column inventory to isolate non-predictive features
print("\nColumn Inventory for Removal Screening:")
non_predictive = [col for col in ['id', 'Unnamed: 32'] if col in df.columns]
print(f"Non-predictive columns flagged for removal: {non_predictive}")

df.isnull().sum():
- Findings on this dataset: The 30 biological nuclear features have zero missing values across all 569 instances. However, the artifact column Unnamed: 32 contains 569 NaN values, indicating it is an empty trailing column generated during CSV export that must be discarded.
- dtype: int64 is printed because columns_with_missing is a pandas Series, and pandas automatically appends its underlying data type to its standard string display.

df.describe:
- Computes the count, mean, standard deviation, minimum, 25th percentile, median (50%), 75th percentile, and maximum across numeric columns.

`missing_counts = df.isnull().sum()` --> Unnamed: 32    569

`non_predictive = [col for col in ['id', 'Unnamed: 32'] if col in df.columns]`

`print(f"Non-predictive columns flagged for removal: {non_predictive}")`  --> Non-predictive columns flagged for removal: ['id', 'Unnamed: 32']

- Confirms that id contains arbitrary identification numbers with no biological relationship to malignancy. Retaining it risks spurious pattern memorization, so both id and Unnamed: 32 must be targeted for removal in Step 3.

In the earlier code block, only df.duplicated().sum() was included to report the count of duplicate rows without directly calling .drop_duplicates().

In [ ]:
# Check shape before removing duplicates
print(f"Shape before removing duplicates: {df.shape}")

# Drop duplicate rows if any exist
df = df.drop_duplicates()

# Check shape after to verify if any rows were removed
print(f"Shape after removing duplicates: {df.shape}")

# Step 3: Define Features and Target

ensuring that the model’s input matrix ($X$) is composed strictly of features that have <mark>biological relevance</mark> to the prediction task, discarding administrative data that would otherwise mislead the training process.
- The patient identification number (id) is an arbitrary database index. Retaining it allows estimators to memorize ID numbers rather than genuine nuclear cell characteristics, leading to catastrophic failure on new patients. The artifact column Unnamed: 32 contains no values and must also be removed.
- In machine learning, predictive input (also called features, independent variables, or $X$) refers to the set of measurable characteristics or signals fed into an algorithm to calculate or predict an unknown outcome.
- Non-Predictive Metadata: These are administrative identifiers or file artifacts (such as patient id number or empty index columns). A patient's ID number contains zero biological information about whether a tumor is cancerous. If included as an input, a complex model might simply memorize that id=842302 is Malignant rather than learning what malignant cell nuclei actually look like.

In [ ]:
import pandas as pd

# Task A: Separate the 30 nuclear diagnostic measurements into input matrix X
# Non-predictive metadata ('id', 'Unnamed: 32') and the target ('diagnosis') are excluded
X = df.drop(columns=['id', 'diagnosis', 'Unnamed: 32'], errors='ignore')

# Task B: Isolate the target column and convert categorical labels to binary integers
# Using .map() as recommended in the lab manual: B -> 0 (negative), M -> 1 (positive)
y = df['diagnosis'].map({'B': 0, 'M': 1})

# Verification
print(f"Feature matrix (X) shape: {X.shape}")  # Expecting (569, 30)
print(f"Target vector (y) shape:  {y.shape}")  # Expecting (569,)
print("\nTarget class distribution (0 = Benign, 1 = Malignant):")
print(y.value_counts())

df.drop(columns=[...]):
- errors='ignore' prevents the cell from crashing if Unnamed: 32 was already dropped in an earlier step.
- The diagnosis column is dropped from the feature matrix $X$ because it is the target variable ($y$) that the model is supposed to predict, not an input feature used to make predictions.
- In pandas, calling df.drop(...) does not modify or delete anything inside df; it only creates and returns a new, separate DataFrame in memory.
- It would only break if you used inplace=True or reassigned back to df
```
# ❌ THIS WOULD CAUSE AN ERROR:
df.drop(columns=['diagnosis'], inplace=True)  # Permanently deletes 'diagnosis' from df!
y = df['diagnosis'].map({'B': 0, 'M': 1})     # KeyError: 'diagnosis' no longer exists in df!
```

#  Step 4: Perform the Train-Test Split First
Perform the Train-Test Split First is considered the most critical methodological step in a machine learning experimental pipeline.
- Preventing Data Leakage: In experimental workflows, preprocessing parameters (such as the mean for missing value imputation or minimum/maximum values for scaling) and feature selection criteria must be learned strictly from the training partition. If these steps are performed before splitting the dataset, information from the test set leaks into the training pipeline, artificially inflating performance metrics and invalidating real-world generalization estimates.


In [ ]:
from sklearn.model_selection import train_test_split

# Partition X and y into 80% training and 20% testing subsets
# stratify=y preserves the ~63%:37% class ratio in both splits
# random_state ensures reproducibility across laboratory runs
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,     # Allocates 20% of the 569 records ($114$ samples) to the test set
    stratify=y,
    random_state=42
)

# Verification
print(f"Training feature set shape (X_train): {X_train.shape}")
print(f"Testing feature set shape  (X_test):  {X_test.shape}")
print(f"Training target vector shape (y_train): {y_train.shape}")
print(f"Testing target vector shape  (y_test):  {y_test.shape}")

print("\nClass distribution in y_train (Proportions):")
print(y_train.value_counts(normalize=True))

print("\nClass distribution in y_test (Proportions):")
print(y_test.value_counts(normalize=True))

stratify=y:
- Ensures that both y_train and y_test mirror the original dataset's class distribution. Without stratification, a random split could accidentally place too many malignant cases in the training set and too few in the test set, skewing evaluation metrics like recall.
<mark>random_state=42</mark>:
- Fixes the pseudorandom seed. This guarantees that every time you restart your Colab runtime and re-run the notebook, the exact same patient indices are assigned to the training and testing sets, ensuring reproducible laboratory results.
- random_state=42 is a widespread convention and cultural standard in the Python machine learning community, though any integer would technically work.
- Passing any fixed integer (like 0, 7, 100, or 42) <mark>stops the random shuffling from changing every time you run the cell</mark>. It freezes the dataset split so your results remain stable and reproducible.

# Step 5: Handle Missing Values
Handle Missing Values addresses incomplete records within the feature matrix before model training.
- missing values in the testing set influence the imputation values injected into the training set (and vice versa). This introduces data leakage.
- The Leakage-Safe Rule: Imputation parameters (like the mean $\mu_{\text{train}}$) must be calculated strictly from the training partition (X_train) and then applied independently to both X_train and X_test.
- Even without any missing values "professional code pipelines must still include an imputer class (such as scikit-learn's SimpleImputer) to <mark>comply with lab rubric requirements</mark> and ensure robust handling of potential missing fields in future deployment data."
  - Preventing Future Leakage during Iteration: If the dataset is updated later—or if you swap it for a noisier clinical dataset with missing entries—having the imputer already wired into the correct sequence (fitted after the train-test split) ensures that your experimental methodology remains leakage-safe by design.
  - Embedding an imputer ensures the pipeline gracefully handles missing data without failing, in the future (real-world data pipelines ingest fresh patient records dynamically; so there could be missing values).
  - Rubric Compliance and Reproducibility: Lab assignments and academic grading rubrics often evaluate students on modular pipeline design. Graders look for standardized <mark>scikit-learn preprocessing chains (Pipeline or explicit transformer steps)</mark> rather than hardcoded assumptions that data will always be pristine.


In [ ]:
# @title
from sklearn.impute import SimpleImputer

# 1. Initialize the imputer to replace missing values (NaN) with the mean of each column
# strategy='mean' is standard for continuous numerical nuclear features
imputer = SimpleImputer(strategy='mean')

# 2. FIT the imputer ONLY on X_train (learns the column means from training data only)
imputer.fit(X_train)

# 3. TRANSFORM both X_train and X_test using the parameters learned from X_train
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Verification
print(f"Missing values in X_train_imputed: {pd.DataFrame(X_train_imputed).isnull().sum().sum()}")
print(f"Missing values in X_test_imputed:  {pd.DataFrame(X_test_imputed).isnull().sum().sum()}")

SimpleImputer(strategy='mean'):
- Instantiates a transformer object that replaces missing NaN values with a descriptive statistic. For continuous biological measurements like nuclear radius and perimeter, mean substitution is standard.
- Other strategies include 'median' (robust to outliers) or 'constant' (filling with $0$).
imputer.transform(...):
- imputer.transform() returns a standard NumPy array rather than a pandas DataFrame. If you want to retain column names and index labels, you can wrap it back into a DataFrame:

#  Step 6: Scale Numerical Features
Scaling places features with different numerical ranges on a comparable scale and is especially important for algorithms sensitive to feature magnitude.

Scaling techniques:
- Min-Max normalization: minimum value ($X_{\min}$) and maximum value ($X_{\max}$)
- $Z$-score standardization: mean ($\mu$) and standard deviation ($\sigma$)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Initialize the Min-Max scaler (bounds features between 0 and 1)
scaler = MinMaxScaler()

# 2. FIT the scaler ONLY on X_train (learns min and max boundaries from training data only)
scaler.fit(X_train_imputed)

# 3. TRANSFORM both X_train and X_test using the parameters learned from X_train
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Verification: Check min and max of scaled training features
import pandas as pd
scaled_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print("Min values across scaled training features (should be 0.0):")
print(scaled_train_df.min().head())

print("\nMax values across scaled training features (should be 1.0):")
print(scaled_train_df.max().head())

MinMaxScaler():
-  Instantiates a normalization transformer object that scales features to a given range (default $[0, 1]$).

Q. "I dont see X_train_imputed used before in any steps. How does scaler.fit(X_train_imputed) work then?"
- In our conceptual walk-through of Step 5, we introduced X_train_imputed to show how imputation connects to scaling, but if you ran Step 4 directly into Step 6 without explicitly executing a standalone imputation cell, X_train_imputed wouldn't exist yet.
- Because the Wisconsin dataset has <mark>zero missing values to begin with, you can actually pass X_train and X_test directly into the scaler without needing an intermediate imputation step</mark>

# Step 7: Perform Exploratory Data Analysis (EDA)
analyzing the training dataset to uncover underlying distributions, feature correlations, and structural patterns before building predictive models.
- EDA must be performed strictly on the training partition (X_train combined with y_train), never on the entire dataset or the test set.
- If we calculate correlation matrices or plot distributions using the whole dataset, information from the test partition leaks into your <mark>feature understanding</mark>, potentially biasing your feature selection choices.

EDA helps answer critical questions:
- Class Balance: Confirming the ~63%:37% Benign-to-Malignant ratio to ensure the model isn't starved of positive (malignant) examples.
- Feature Collinearity: Identifying highly correlated features (e.g., radius, perimeter, and area measure similar geometric properties of cell nuclei), which helps explain why certain feature selection or regularization techniques are needed later.
- Separability: Discovering which individual <mark>morphometric features</mark> exhibit the clearest mathematical boundary between benign and malignant tumors.

"morphometric features": In the context of the Wisconsin Diagnostic Breast Cancer (WDBC) dataset, morphometric features are quantitative measurements of the physical shape, size, and structural geometry of cell nuclei extracted from digitized fine-needle aspirate (FNA) biopsy images.
- For each physical property, the dataset records three distinct mathematical variations computed across all nuclei found in a sample:
    1.  **Mean (`_mean`):** The average value of the measurement (e.g., average cell radius).
    2.  **Standard Error (`_se`):** The standard error or variability of the measurement.
    3.  **"Worst" (`_worst`):** The mean of the three largest/most extreme values observed, which often captures the most aggressive cell clusters.
- Cancerous (malignant) cell nuclei tend to be larger, more irregular in shape (low smoothness/compactness), and exhibit higher variation compared to healthy (benign) cells. These 30 morphometric inputs serve as the mathematical signals the machine learning model uses to distinguish between benign and malignant tissue.

"Feature Understanding": Feature understanding (often referred to in data science as feature analysis or domain-driven inspection) is the process of examining what each input variable measures, how it behaves statistically, and why it correlates with the target outcome. Rather than treating features as abstract numbers in a <mark>blind matrix ($X$)</mark>, feature understanding connects data columns back to real-world physical or biological principles.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Recombine X_train_scaled and y_train into a single training DataFrame for EDA
# (Always use training data only to prevent data leakage!)
train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
train_df['diagnosis'] = y_train.values

# 1. Class Distribution Analysis
print("Training Class Distribution Counts:")
print(train_df['diagnosis'].value_counts())

plt.figure(figsize=(5, 3))
sns.countplot(x='diagnosis', data=train_df, palette='Set2')
plt.title('Distribution of Benign (0) vs Malignant (1) in Training Set')
plt.xlabel('Diagnosis')
plt.ylabel('Patient Count')
plt.show()

print("\n\n\n")

# 2. Correlation Matrix Heatmap (Checking for Multicollinearity among features)
plt.figure(figsize=(10, 8))
corr_matrix = train_df.iloc[:, :10].corr() # Inspecting first 10 mean features for readability
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap (First 10 Mean Nuclear Features)')
plt.show()

### # Functions used:
value_counts():
- Validates the exact count of benign ($0$) and malignant ($1$) samples in your training split.
- Used to print the exact numerical counts of Benign ($0$) vs. Malignant ($1$) samples in the training set.

sns.countplot():
- Used to plot the categorical bar chart of the class distribution.

.corr():
- Used to compute Pearson's correlation coefficient matrix across the numerical feature

columns.sns.heatmap():
- Used to visualize the correlation matrix as a color-coded heatmap.

---

### # Funtions not used:
- sns.histplot()
- sns.boxplot()
- sns.violinplot()

Why Aren't They There, and How Do You Use Them?
- The three unused functions—sns.histplot(), sns.boxplot(), and sns.violinplot()—were omitted from the core code snippet not because they are unhelpful, but due to scope, conciseness, and the specific questions we needed to answer during Step 7.
- While count plots and heatmaps cover class balance and multicollinearity, the other three functions are standard tools for examining feature distributions and how strongly individual features separate benign from malignant tumors.
- reasoning behind why they were left out of that specific walkthrough:
    - The "Plot Explosion" Problem (Feature Overload): The Wisconsin Diagnostic Breast Cancer dataset contains 30 distinct numerical features (radius_mean, texture_mean, area_worst, etc.). If you run a box plot or a histogram for every single feature individually, your Colab notebook will generate 30 separate charts. While thorough, this creates a massive wall of images that makes the notebook difficult to read and grade.
    - The core snippet focused strictly on the two highest-priority exploratory questions for this specific pipeline: Class Balance (sns.countplot), Multicollinearity (sns.heatmap)

---

### # What Each Unused Function Does (And When You Would Use It)
- sns.histplot() (Univariate Distribution): ***What it checks:*** Whether a feature's values are normally distributed, heavily skewed, or multi-modal, often overlaying a Kernel Density Estimate (KDE). ***Why it was skipped:*** While great for checking distribution shapes, it doesn't immediately show how well a feature separates Benign from Malignant classes unless you add a hue='diagnosis' argument.
- sns.boxplot() (Outliers and Interquartile Range): ***What it checks:*** The median, 25th/75th percentiles, and statistical outliers (plotted as individual dots beyond the whiskers). ***Why it was skipped:*** In clinical datasets, extreme values might represent genuinely aggressive malignant tumors rather than data entry errors. Unless you plan to drop or cap outliers, running a box plot for all 30 features is usually saved for a dedicated data-cleaning phase rather than basic EDA.
- sns.violinplot() (Probability Density & Spread): ***What it checks:*** Combines a box plot with a kernel density trace on either side, showing the full shape of the data distribution across classes. ***Why it was skipped:*** Violin plots are visually rich and excellent for showing how cleanly a single feature (like radius_mean) separates the two classes. However, they are computationally heavier and take up more vertical space in a notebook than a simple summary stat or correlation matrix.


### <mark>Seaborn plotting command</mark>
palette='Set2':
- ColorBrewer Standard: 'Set2' is one of the classic qualitative color palettes originally designed by ColorBrewer and built directly into Matplotlib and Seaborn.
- It provides a set of soft, muted, and visually distinct colors (typically featuring soft greens, blues, oranges, and pinks). It avoids harsh, high-saturation neon colors, making charts look professional and clean for academic reports or lab presentations.

Palette types:
- Qualitative / Categorical Palettes (Best for discrete classes like Benign vs. Malignant):
    - 'Set2' (Soft, muted qualitative palette)
    - Set1' or 'Set3' (Brighter or alternative categorical variations)
    - 'deep', 'muted', 'pastel', 'bright', 'dark', 'colorblind' (Seaborn's default qualitative themes; 'deep' is the default if no palette is specified).
- Sequential Palettes (Best for continuous values or ordered data, like heatmaps): These progress from a light color to a dark color (e.g., 'Blues', 'viridis', 'magma', 'YlGnBu').
- Custom Color Lists (Full Control)


# Feature Selection Steps
Steps 8 through 15 are entirely dedicated to feature selection processes. They represent different mathematical philosophies and techniques used to filter, rank, and reduce the 30 original diagnostic features down to smaller, more optimized subsets before model training.
-   **Step 8 (Correlation-Based):** Removes redundancy by dropping highly correlated pairs of features.
-   **Step 9 (Chi-Square):** Ranks features using statistical association tests relative to the target.
-   **Step 10 (RFE):** Uses a model-driven wrapper approach to iteratively eliminate weak features.
-   **Step 11 (RFECV):** Extends RFE by incorporating cross-validation to guide the optimal number of features.
-   **Step 12 & 13 (Random Forest & Extra Trees Importances):** Uses tree-based ensemble models to score and rank feature contributions.
-   **Step 14 (L1-Based Selection):** Uses L1 regularization (Lasso) in Logistic Regression to shrink useless feature coefficients down to zero.
-   **Step 15 (Voting-Based Selection):** Combines the votes/rankings from multiple selection methods to identify consensus features that repeatedly appear important.

The primary reasons for exploring so many different techniques include:

-   **Different Mathematical Perspectives:** Each method evaluates data through a completely different lens. For example, correlation checks for linear redundancy, Chi-Square tests statistical independence, tree-based models (Random Forest/Extra Trees) capture non-linear splits, and L1 regularization uses coefficient shrinkage. Because you cannot know beforehand which relationship matters most to your model, testing multiple lenses prevents you from missing key patterns.
-   **Enabling Consensus (Voting-Based Selection):** Running a wide variety of methods directly enables **Step 15 (Voting-Based Feature Selection)**. By tallying votes across multiple techniques, you can identify "consensus features" that repeatedly prove their value regardless of the algorithm used.
-   **Matching Features to Classifiers:** Different machine learning models (such as Logistic Regression, Support Vector Machines, or Decision Trees) interact differently with input features. Comparing multiple feature subsets helps determine which specific combination of feature selector and classifier yields the highest performance.
-   **Empirical Justification:** In a scientific or lab setting, you cannot rely on guesswork. The laboratory workflow requires you to build a final comparison table across all methods to empirically prove which subset achieves the best test-set accuracy, precision, recall, and ROC-AUC.

# Step 8: Correlation-Based Feature Selection
Correlation-Based Feature Selection addresses feature redundancy by identifying and removing highly correlated input variables before model training.
- The Problem of Multicollinearity: As observed in the <mark>correlation matrix during Step 7</mark>, variables like radius_mean, perimeter_mean, and area_mean measure nearly identical geometric properties of cell nuclei. Including all three introduces <mark>collinearity (redundant information)</mark>, which can destabilize linear models, inflate coefficient variances, and increase computational overhead without adding predictive value.
- The Selection Strategy: A correlation-based feature selector scans the upper triangle of the training feature correlation matrix. When it finds a pair of features with a linear correlation coefficient ($r$) exceeding a predefined threshold (e.g., $r > 0.90$), it <mark>drops the feature that carries less independent variance or is more redundant</mark>.
- The Leakage-Safe Rule: Like scaling and imputation, feature selection thresholds and dropped columns must be computed strictly from the training set (X_train) and then applied identically to the testing set (X_test). If you evaluate correlations across the entire dataset, information from the test set leaks into your selection logic.

In [ ]:
import numpy as np
import pandas as pd

# Ensure train_df is constructed from X_train_scaled (training data only!)
train_features = pd.DataFrame(X_train_scaled, columns=X.columns)

# 1. Compute the correlation matrix on training features
corr_matrix = train_features.corr().abs()

# 2. Select the upper triangle of the correlation matrix
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 3. Identify features with a correlation greater than the threshold (e.g., 0.90)
threshold = 0.90
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

print(f"Features flagged for removal due to high collinearity (r > {threshold}):")
print(to_drop)

# 4. Drop the redundant features from both X_train_scaled and X_test_scaled
X_train_selected = train_features.drop(columns=to_drop)
X_test_selected = pd.DataFrame(X_test_scaled, columns=X.columns).drop(columns=to_drop)

print(f"\nOriginal feature count: {X_train_scaled.shape[1]}")
print(f"Reduced feature count after selection: {X_train_selected.shape[1]}")

corr_matrix.abs():
- A strong negative correlation ($r = -0.95$) indicates just as much redundancy and linear dependency as a strong positive correlation ($r = 0.95$). Taking the absolute value ensures both directions are flagged.

np.ones(shape):
- This NumPy function creates a brand-new matrix filled entirely with the number 1 (or 1.0), shaped according to whatever tuple you pass into it.

"k=1":
- np.triu stands for numpy triangle upper. The k parameter controls where the diagonal dividing line is drawn relative to the main center diagonal:
- k = 0 (Default): Draws the dividing line right down the center diagonal. It keeps the main diagonal (the 1.0 self-correlations) and everything above it.
- k = 1 (Shifted Up): Shifts the dividing line one step above the main diagonal.

.astype(bool):
- NumPy and pandas method used to explicitly convert (or "cast") the data type of an array or DataFrame into booleans (True or False).

"what is the upper triangle?"
```
        Columns (Features)
        ┌───────┬───────┬───────┐
Rows    │  1.0  │  0.95 │  0.80 │  <-- Upper Triangle (Unique pairs)
        ├───────┼───────┼───────┤
        │  0.95 │  1.0  │  0.65 │  <-- Diagonal (1.0) &
        ├───────┼───────┼───────┤      Lower Triangle (Duplicates)
        │  0.80 │  0.65 │  1.0  │
        └───────┴───────┴───────┘
```
- In a square correlation matrix, the upper triangle refers to all the cells located above the diagonal line that runs from the top-left to the bottom-right of the matrix.

# Step 9: Chi-Square Feature Selection
a statistical filter method used to <mark>select the most informative features </mark> by evaluating the independence between each individual feature and the target class ($y$).
- The Core Objective: The Chi-Square ($\chi^2$) test measures how much the observed distribution of a feature deviates from what we would expect if the feature and the target diagnosis were completely independent.
- The Leakage-Safe Rule: Just like scaling and correlation filtering, the Chi-Square selector must be fitted strictly on the training partition (X_train and y_train). Scoring features using the entire dataset would leak test set distributions into your feature selection logic.

<h3 align=center><mark>The Crucial Prerequisite: Non-Negative Values</mark></h3>

Why? The Chi-Square formula compares observed frequencies/values against expected values. Negative numbers break this statistical interpretation.



In [ ]:
from sklearn.feature_selection import SelectKBest, chi2
import pandas as pd

# 1. Initialize SelectKBest with the chi2 scoring function and choose K features (e.g., top 15)
k_best_features = 15
chi2_selector = SelectKBest(score_func=chi2, k=k_best_features)

# 2. FIT strictly on the scaled training data and training labels
chi2_selector.fit(X_train_scaled, y_train)

# 3. TRANSFORM both training and testing datasets
X_train_chi2 = chi2_selector.transform(X_train_scaled)
X_test_chi2 = chi2_selector.transform(X_test_scaled)

# 4. Identify which specific features were kept
selected_mask = chi2_selector.get_support()
selected_feature_names = X.columns[selected_mask]

print(f"Original feature count: {X_train_scaled.shape[1]}")
print(f"Reduced feature count after Chi-Square: {X_train_chi2.shape[1]}")
print("\nSelected Features:")
print(list(selected_feature_names))

SelectKBest(score_func=chi2, k=15):
- Acts as a wrapper that takes the scores from the chi2 function and automatically ranks all features, retaining only the top $k$ features (in this example, 15 out of 30) while dropping the rest.

<mark>Chi-Square vs. ANOVA for WDBC</mark>:
- While chi2 is a standard curriculum exercise for teaching statistical filter methods, data science instructors often point out a minor technical caveat regarding the Wisconsin Breast Cancer dataset:
- Chi-Square is traditionally designed for categorical or frequency count data.
- Because the WDBC morphometric features are continuous measurements, scikit-learn's ANOVA F-value (f_classif) is theoretically more appropriate for continuous variables.

# Step 10: Recursive Feature Elimination (RFE)
an advanced wrapper-style feature selection method designed to find the optimal subset of <mark>predictive features</mark> by iteratively training a machine learning model.
- In previous steps (Chi-Square and Correlation), we used filter methods, which evaluate features independently based on statistical scores or matrix thresholds without training an actual model.
- RFE is a wrapper method. It actively uses a machine learning algorithm (like Logistic Regression) to evaluate how well features work together, making it more powerful and tailored to the specific model you intend to use.

How RFE Works Iteratively:
- Train: The algorithm fits a base model (estimator) using all available features.
- Rank: It inspects the model's coefficients or feature importances to rank every feature from most to least important.
- Prune: It <mark>permanently removes</mark> the single weakest (least important) feature.
- Repeat: It loops through this process recursively until the target number of features ($k$) is reached.

"permanently removes"?
- it does not touch, modify, or permanently alter your original dataset (df or raw X) stored in memory or on disk. When data science tutorials or documentation use phrases like "permanently removes" or "drops," it means the feature is removed from that specific working subset (e.g., X_train_rfe) for the remainder of the pipeline.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pandas as pd

# 1. Define the base estimator that RFE will use to judge feature importance
# (Logistic Regression coefficients act as feature weights)
estimator = LogisticRegression(max_iter=1000, random_state=42)

# 2. Initialize RFE to target a specific number of features (e.g., top 10)
num_features_to_select = 10
rfe_selector = RFE(estimator=estimator, n_features_to_select=num_features_to_select, step=1)

# 3. FIT strictly on the scaled training data and training labels
rfe_selector.fit(X_train_scaled, y_train)

# 4. TRANSFORM both training and testing datasets
X_train_rfe = rfe_selector.transform(X_train_scaled)
X_test_rfe = rfe_selector.transform(X_test_scaled)

# 5. Extract the selected feature names
selected_mask = rfe_selector.support_
selected_feature_names = X.columns[selected_mask]

print(f"Original feature count: {X_train_scaled.shape[1]}")
print(f"Reduced feature count after RFE: {X_train_rfe.shape[1]}")
print("\nSelected Features via RFE:")
print(list(selected_feature_names))

#  Step 11: Recursive Feature Elimination with Cross-Validation (RFECV)
While regular RFE requires you to manually guess or predefine the exact number of features you want to keep (e.g., n_features_to_select=10), RFECV solves this by letting cross-validation automatically find the optimal number of features.
- RFECV performs recursive feature elimination across multiple cross-validation folds. At each step of dropping features, it measures the validation performance. It then identifies the exact subset size that maximizes generalization performance.
- Instead of arbitrarily choosing to keep 10 or 15 features, RFECV uses cross-validation to test how well different feature subset sizes actually perform on <mark>unseen validation</mark> data before touching the final test set.

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import pandas as pd

# 1. Define the base estimator
estimator = LogisticRegression(max_iter=1000, random_state=42)

# 2. Setup Stratified Cross-Validation for the inner loop
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Initialize RFECV
rfecv_selector = RFECV(
    estimator=estimator,
    step=1,
    cv=cv_strategy,
    scoring='accuracy' # or 'roc_auc'
)

# 4. FIT strictly on the training data
rfecv_selector.fit(X_train_scaled, y_train)

# 5. Transform training and test sets using the optimal feature count found
X_train_rfecv = rfecv_selector.transform(X_train_scaled)
X_test_rfecv = rfecv_selector.transform(X_test_scaled)

print(f"Optimal number of features chosen by RFECV: {rfecv_selector.n_features_}>")
print(f"Selected Features: {list(X.columns[rfecv_selector.support_])}")

.support_:
- A boolean mask attribute used to identify which final features survived the cross-validated elimination process.

.n_features_:
- An attribute automatically populated after fitting that reveals the optimal number of features discovered by the cross-validation scoring.

# Step 12: Random Forest Feature Importance
it shifts away from linear correlation and statistical frequency tests, introducing an <mark>ensemble tree-based approach</mark> to feature selection.